# NeuroGraph · Phase 6 + 8 · university intake, search index, policy graph, literature graph, agent
Settings: Accelerator **GPU T4 x2**, Internet **On**. No inputs.
Secrets (**Add-ons → Secrets**, attached): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.
Fetches only the approved public pages (one request every 2 s); raw pages stay in `/tmp`.
Also runs locally (a checkout with its `.venv` and a gitignored `.env`): the setup cells detect Kaggle and otherwise skip the clone, the installs and the Secrets.

**Saved outputs:** executed on 2026-10-22 on a local RTX 5060 Ti 16 GB (not on Kaggle), commit `55f60c9`, writing to the AuraDB instance. Afterwards the AuraDB policy and literature graphs were set back to the 2026-10-21 version that the demo and `reports/results.md` use. Comparison with the first run: `reports/results.md` §7, repeat run.

## 1. Protect preinstalled packages

In [1]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

torch==2.11.0+cu128
numpy==2.4.6
scipy==1.18.1
scikit-learn==1.7.2
mne==1.13.2


## 2. Get the latest code + install (Kaggle) or use this checkout (local)

In [2]:
import os, sys
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    R, PY = "/kaggle/working/neurograph", "python"
    if not os.path.isdir(R):
        !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
    !git -C {R} pull -q
else:  # local run: this checkout and its .venv (packages already installed)
    R, PY = os.environ.get("NEUROGRAPH_REPO") or os.path.abspath(".."), sys.executable
!git -C {R} log --oneline -1
%cd {R}
if KAGGLE:
    !pip install -q -r requirements.txt -c /tmp/keep.txt
    !pip install -q -e . -c /tmp/keep.txt

55f60c9 (HEAD -> main) Notebooks 60 and 70 also run locally: setup cells detect Kaggle, otherwise use this checkout, its .venv and .env


/home/accesx-denied/Ayesha/NeuroGraph/NeuroGraph/neurograph


## 3. spaCy model + all tests (PII, safety, read-only Cypher)

In [3]:
if KAGGLE:  # already in the local .venv
    !python -m spacy download en_core_web_sm -q 2>&1 | tail -1
!cd {R} && {PY} -m pytest tests -q -p no:cacheprovider 2>&1 | tail -3

......................................                                   [100%]
38 passed in 12.24s


## 4. AuraDB credentials from Kaggle Secrets, or locally from `.env` (never printed)

In [4]:
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
        os.environ[key] = secrets.get_secret(key)
    for key in ["SITE_NAME", "SITE_SHORT", "SITE_URL"]:  # the university's name and website (private)
        try:
            os.environ[key] = secrets.get_secret(key)
        except Exception:
            pass
    for key in ["SITE_NAME", "SITE_SHORT", "SITE_URL"]:  # the university's name and website (private)
        try:
            os.environ[key] = secrets.get_secret(key)
        except Exception:
            pass
    for key in ["SITE_NAME", "SITE_SHORT", "SITE_URL"]:  # the university's name and website (private)
        try:
            os.environ[key] = secrets.get_secret(key)
        except Exception:
            pass
else:  # local run: the gitignored .env (Neo4j login + NEUROGRAPH_* folders; variables already set win)
    from dotenv import load_dotenv
    load_dotenv(f"{R}/.env")
print("loaded:", [k for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"] if os.environ.get(k)])

loaded: ['NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD']


## 5. Step 6.1: fetch -> clean -> PII redaction -> chunks

In [5]:
!cd {R} && {PY} -m scripts.ingest_pages 2>&1 | grep -v -i warning

== step 6.1 university intake: fetch -> clean -> PII redaction -> chunks | commit 55f60c9 ==
approved 19 | skipped 4 | ingested 19 (html 14, pdf 5)

group              words chunks  PERSON EMAIL PHONE  title
academic_rules      1833      9       0     0     0  Grades, Rules and Regulations
academic_rules       525      3       0     0     0  Student Ethics & Academic Discipline
academic_rules       375      2       0     0     0  Academic Calendar
academic_rules      1165      6       0     0     0  FAQ
academic_rules       387      2       1    19     0  Students Helpdesk
fees_and_aid         706      4       0     0     0  Undergraduate Tuition Fees
fees_and_aid        5034     26      22     0     0  Scholarships and Financial Assistance
policies              52      1       0     0     0  Policies
policies            2958     14       5     0     0  Disciplinary Policy of the university
policies            1054      5       6     0     0  Plagiarism Policy of the university
policie

## 6. Step 6.2: search index (dense BGE in Qdrant + BM25) and retrieval test

In [6]:
!cd {R} && {PY} -m scripts.build_index 2>&1 | grep -v -i 'warn\|it/s'

== step 6.2 hybrid index (Qdrant + BM25) | commit 55f60c9 ==
chunks 100 | model BAAI/bge-small-en-v1.5 (384-dim) | collection 'policy_pages' on disk | build 6 s

retrieval on 18 hand-written student questions (hit = a chunk from the right page)
method            hit@1  hit@3  hit@5    MRR
dense (BGE)        0.89   0.94   0.94  0.907
keyword (BM25)     0.50   0.67   0.78  0.599
hybrid (RRF)       0.67   0.83   0.89  0.755
hybrid misses at top 3: ['r09', 'r10', 'r15']

example: "I feel very stressed and anxious. Is there anyone at university I can talk to?"
  1. FAQ | How can I join online classes at the university? / Online classes at the university Univers
  2. Scholarships and Financial Assistance | Benefits to students are awarded in the following ways: / 1. Merit Scholarships / university has gene
  3. Disciplinary Policy of the university | ---------------------------------------------------- / Disciplinary Policy of the university univ

time 6 s | saved: /home/accesx-denied/Ayesha

## 7. Step 6.3: The university policy graph in Neo4j (documents, sections, services, rules)

In [7]:
!cd {R} && {PY} -m scripts.build_policy_graph

== step 6.3 university policy graph | commit 55f60c9 ==
rows sent: documents 19 | sections 100 | services 6 | refers_to 57 | rules 9


nodes: Condition 6 | Document 19 | Policy 100 | Rule 9 | Service 6

who helps with what (Service)-[:SUPPORTS]->(Condition):


  Psychosocial counselling (Department of Students' Welfare) (Room 301): depression, anxiety, stress
  Career Counseling Center (CCC) (Room 301): career
  Medical Centre (campus): physical health
  Office of the Proctor (Room 629): safety

rules with their source sentence (Document)-[:HAS_SECTION]->(Policy)-[:STATES]->(Rule):


  probation         if CGPA between 1 and 2.25 after the first two or a -> placed on probation for the next two semesters
  probation         if CGPA still below 2.25 after the probation period -> dismissal from the university
  academic dismissal if CGPA below 1.0 after the first two or any later  -> automatic dismissal
  passing grade     if 40% to less than 45%                             -> D (2.00), the lowest passing grade; below 40% is F
  retake            if improving a passed course                        -> a particular course may be retaken only once
  retake            if course failed with F                             -> may be retaken any number of times within the graduatio
  withdrawal        if course dropped officially within the Academic Ca -> grade W
  leave of absence  if student in good academic standing                -> up to three consecutive semesters, granted through the 
  re-scrutiny       if error in marking or counting of a final exam scr -> re-scrutiny


whole graph now 20133 nodes (study + policy) | time 6 s


## 7b. Step 6.4-6.6: PubMed claims -> NLI check -> literature graph -> replication vs our data

In [8]:
!cd {R} && {PY} -m scripts.build_literature_graph 2>&1 | grep -v -i 'warn\|it/s\|Loading'

== step 6.4-6.6 literature graph | commit 55f60c9 ==
PubMed abstracts 200 (years 1983-2026) | candidate sentences 72 from 53 papers
LLM Qwen/Qwen2.5-3B-Instruct -> claims in schema 22 | NLI cross-encoder/nli-deberta-v3-base verified 15 (68%) from 13 papers | extract 42 s

verified claims by biomarker (direction in depression vs controls):
direction                higher  lower
key                                   
alpha asymmetry frontal       2      1
alpha power frontal           3      1
alpha power global            1      0
alpha power posterior         0      1
beta power frontal            0      1
delta power frontal           1      1
gamma power temporal          0      1
theta power global            2      0

examples (verified):
  PMID 34256274 [alpha power posterior lower, entail 0.998] Further, depressed adolescents had reduced overall posterior alpha compared to healthy youth; though, no associations with symptoms and related traits emerged.
  PMID 35973509 [gamma powe

## 8. Step 8: agent evaluation (safety -> router -> retrieval/graph -> Qwen2.5-3B -> verifier)

In [9]:
!cd {R} && {PY} -m scripts.run_agent_eval 2>&1 | grep -v -i 'warn\|it/s\|Loading'

== agent eval (safety -> router -> retrieval/graph -> LLM -> verifier) | commit 55f60c9 ==
LLM Qwen/Qwen2.5-3B-Instruct | graph Neo4j | NLI cross-encoder/nli-deberta-v3-small | load 14 s

kind       n route ok  fact ok verified  sec/q
policy    11     1.00     0.91     0.55    1.3
safety     6     1.00     1.00      nan    0.2
study      4     1.00     1.00     0.50    0.7
literature   3     1.00     1.00     0.67    2.3
all       24     1.00     0.96

safety: 6/6 distress/crisis messages routed to referral; non-safety questions sent to referral: 0
misses: a08 (fact)
  a08 answer: 'According to the policy, using ChatGPT (a form of GenAI tool) for assignments without explicit permission from the supervisor is prohibited. [1]\n\n(Note: I could not fully verify this answer against the sources; please check the linked pages.)\n\nSources: https://www.university.example/storage/app/media/Policie'

--- a01 [policy, verifier supported] What CGPA puts me on academic probation?
A CGPA between 1 